# Clara · Colab client

All logic lives in the `clara` package. This notebook only installs it and calls it, so there's nothing to re-run cell by cell.

Replace `<you>` with your GitHub username after you push the repo.

In [ ]:
!pip install -q "clara[studio] @ git+https://github.com/<you>/clara-studio.git"

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# os.environ["CLARA_PROVIDER"] = "mock"   # uncomment to run without a key

## Run a single episode

In [ ]:
from clara.episode import run_episode
run_episode("clara-moderate", "pharmacy-refill",
            "Give one step per message, in plain words, and confirm before moving on.",
            "one-step", verbose=True)

## Compare styles

In [ ]:
import pandas as pd
STYLES = {
    "baseline-thorough": "Be thorough. Explain every option and all the details in each message.",
    "one-step": "Give one step per message, in plain words, and confirm before moving on.",
}
rows = []
for label, style in STYLES.items():
    for persona in ["clara-mild", "clara-moderate"]:
        r = run_episode(persona, "pharmacy-refill", style, label)
        rows.append({"style": label, "persona": persona, **r["score"]})
pd.DataFrame(rows)

## Hand the experiment to the research agent

In [ ]:
from clara.agent_tools import new_agent
lead = new_agent()
OBJECTIVE = (
    "Find a communication style for the pharmacy-refill assistant that lets clara-moderate "
    "succeed without abandoning. Start from the baseline style 'Be thorough. Explain every option and all the "
    "details in each message.' Use clara-mild as a control. Try at most 4 variants."
)
print(lead.run(OBJECTIVE))

## Review everything saved so far

In [ ]:
from clara import storage
pd.DataFrame([{"file": e["_file"], **e["score"]} for e in storage.load_episodes()])

## Open the designer studio from Colab (temporary public link)

In [ ]:
from clara.studio import build_demo, launch_kwargs
build_demo().queue().launch(share=True, **launch_kwargs())